# 25 - Step 4 open-set gate (pre-registered)

Runs the approved Step 4 protocol (`paper/cvpr2027/STEP4_PROPOSAL.md`, revision 2):
a supervised-contrastive head on frozen DINOv2 features, class prototypes, and a
calibrated threshold that routes detections the detector should not trust to the
VLM. Detector training and evaluation stay in notebooks 05/06/07; this notebook
only does the gate.

**Stages** (set `STAGE`, run all):

| STAGE | milestone | runtime | what |
|---|---|---|---|
| `gt_features` | M2 | GPU | ground-truth crops of every rotation's train/val split -> DINOv2 features |
| `val_features` | M2 | GPU | selected detector crops of each rotation's **validation** split (notebook 07 `EVAL_SPLIT="val"`) |
| `train` | M3 | CPU ok | train the 5 heads, prototypes, calibration |
| `test_features` | M3 | GPU | selected detector crops of the evaluated **test** frames |
| `evaluate` | M3 | CPU | all arms, endpoints P1–P3 / S1a–S1b and the GO decision — **run once** |

M3 stages refuse to run until `M2_SIGNED_OFF = True` (set only after the evaluator's
M2 sign-off is recorded in the proposal). Every stage writes a `meta.json` with file
hashes for the change log (§10).


In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    REPO_URL = "https://github.com/Ali-RT/vision-uss-research"
    if not (REPO_DIR / ".git").exists():   # fresh VM: clone (no separate bootstrap script)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    # always run the latest main; discards VM-local edits to tracked repo files
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", "origin/main"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import subprocess, sys

STAGE = "gt_features"        # gt_features | val_features | train | test_features | evaluate
M2_SIGNED_OFF = False        # True only after the evaluator signs off M2 (proposal §10)
GATE_RUN = "gate_v1"         # outputs under ARTIFACTS/gate/<GATE_RUN>/
SMOKE_LIMIT = 0              # >0: tiny smoke run into a separate *_smoke dir (never a protocol output)

ROTATIONS = {                # rotation -> (dataset run, held-out types, eval tag)
    "woodenboard": ("dataset_v2holdwoodenboard", "woodenboard", "holdwoodenboard"),
    "speedbump": ("dataset_v2holdspeedbump", "speedbump", "holdspeedbump"),
    "bush": ("dataset_v2holdbush", "bush", "holdbush"),
    "bollard": ("dataset_v2holdbollard", "bollard", "holdbollard"),
    "treecubehose": ("dataset_v2holdtreecubehose", "tree,cubestandard,hose", "holdtreecubehose"),
}
IMAGES_FROM = "dataset_v2holdwoodenboard"   # any rotation's tar holds every labelled frame

if STAGE in ("train", "test_features", "evaluate") and not M2_SIGNED_OFF:
    raise SystemExit(f"STAGE={STAGE} is M3: blocked until the evaluator signs off M2")

GATE_DIR = ARTIFACTS / "gate" / (GATE_RUN + ("_smoke" if SMOKE_LIMIT else ""))
GATE_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ARTIFACTS / "datasets"
REPORTS = ARTIFACTS / "reports"

# images: unpack one rotation's staged tar to local disk (fast), as in notebooks 06/07
local = Path("/content/dataset") / IMAGES_FROM
tar = DATASETS / IMAGES_FROM / "local_images.tar"
if STAGE not in ("train", "evaluate") and not (local / "images").exists():
    assert tar.exists(), f"missing {tar}: run notebook 06 STAGE_ONLY for {IMAGES_FROM}"
    local.parent.mkdir(parents=True, exist_ok=True)
    print(f"unpacking {tar.name} ...")
    subprocess.run(["tar", "-xf", str(tar), "-C", str(local.parent)], check=True)
SCRIPT = [sys.executable, str(REPO_DIR / "scripts" / "gate_features.py")]
LIMIT = ["--limit", str(SMOKE_LIMIT)] if SMOKE_LIMIT else []
print(f"stage {STAGE} -> {GATE_DIR}")

In [ ]:
# ---- M2: ground-truth crops (train/val of every rotation) -> features
if STAGE == "gt_features":
    rot = [f"{r}={DATASETS / d / 'dataset_manifest.csv'}" for r, (d, _, _) in ROTATIONS.items()]
    held = [f"{r}={h}" for r, (_, h, _) in ROTATIONS.items()]
    subprocess.run(SCRIPT + ["gt", "--images", str(local), "--rotation", *rot, "--heldout", *held,
                             "--reserved", str(REPO_DIR / "configs/splits/reserved_sessions_v1.csv"),
                             "--out", str(GATE_DIR / "gt"), *LIMIT], check=True)

In [ ]:
# ---- M2: selected detector crops of each rotation's VALIDATION split (calibration)
if STAGE == "val_features":
    for r, (d, _, tag) in ROTATIONS.items():
        rep = REPORTS / f"eval_{tag}_val"
        if not (rep / "inference_status.csv").exists():
            print(f"skip {r}: run notebook 07 with MODEL_RUN=train_v1_{tag}, EVAL_TAG={tag!r}, EVAL_SPLIT='val'")
            continue
        subprocess.run(SCRIPT + ["det", "--images", str(local), "--class-names-from", str(DATASETS / d),
                                 "--raw", str(rep / "raw_predictions.csv"),
                                 "--status", str(rep / "inference_status.csv"),
                                 "--out", str(GATE_DIR / f"det_{r}_val"), *LIMIT], check=True)

In [ ]:
# ---- M3 (after M2 sign-off): train heads, prototypes, calibration
if STAGE == "train":
    for r, (d, h, _) in ROTATIONS.items():
        subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "gate_train.py"), "--rotation", r,
                        "--gt", str(GATE_DIR / "gt"), "--det", str(GATE_DIR / f"det_{r}_val"),
                        "--manifest", str(DATASETS / d / "dataset_manifest.csv"), "--heldout", h,
                        "--out", str(GATE_DIR / f"gate_{r}")], check=True)

In [ ]:
import json as _json
# ---- M3 (after M2 sign-off), run ONCE: every arm, endpoints P1-P3 / S1a-S1b, GO
if STAGE == "evaluate":
    M = ARTIFACTS / "method"
    TIER_D_VLM = [M / r / "scored.csv" for r in ("prompt_decision_v1", "prompt_decision_low_v1",
                                                 "prompt_decision_high_v1")]
    cfg = {"rotations": {}, "dev_low_types": ["woodenboard", "speedbump"]}
    for r, (d, h, tag) in ROTATIONS.items():
        tier_c = r == "treecubehose"
        cfg["rotations"][r] = {
            "tier": "C" if tier_c else "D", "heldout": h.split(","), "dataset": str(DATASETS / d),
            "gate": str(GATE_DIR / f"gate_{r}"), "det_test": str(GATE_DIR / f"det_{r}_test"),
            "vlm_scored": [str(M / "prompt_decision_tierc_v1" / "scored.csv")] if tier_c else list(map(str, TIER_D_VLM)),
            "vlm_fixed": str(M / ("prompt_decision_tierc_fixed_v1" if tier_c else "prompt_decision_fixedcrop_v1")
                             / "scored.csv"),
            "low_extra": ["hose"] if tier_c else []}
    (GATE_DIR / "evaluate_config.json").write_text(_json.dumps(cfg, indent=1))
    subprocess.run([sys.executable, str(REPO_DIR / "scripts" / "gate_evaluate.py"),
                    "--config", str(GATE_DIR / "evaluate_config.json"),
                    "--out", str(GATE_DIR / "evaluation")], check=True)

In [ ]:
# ---- M3 (after M2 sign-off): selected detector crops of the evaluated TEST frames
# fixed at M2 (hashes in the proposal's §10): tier D = the step-3 frames in each
# model's own test split; tier C = scripts/tierc_frames.py on the tier C manifest
TEST_FRAMES = {r: REPO_DIR / "configs" / "step4" / f"tierd_frames_{r}.csv"
               for r in ("woodenboard", "speedbump", "bush", "bollard")}
TEST_FRAMES["treecubehose"] = REPO_DIR / "configs" / "step4" / "tierc_frames_treecubehose.csv"
if STAGE == "test_features":
    for r, (d, _, tag) in ROTATIONS.items():
        rep = REPORTS / f"eval_{tag}"
        assert r in TEST_FRAMES, f"{r}: frame manifest not set"
        subprocess.run(SCRIPT + ["det", "--images", str(local), "--class-names-from", str(DATASETS / d),
                                 "--raw", str(rep / "raw_predictions.csv"), "--frames", str(TEST_FRAMES[r]),
                                 "--out", str(GATE_DIR / f"det_{r}_test")], check=True)
print("done:", sorted(p.name for p in GATE_DIR.iterdir()))